## 02 - Benchmark Model
- Use the previous 24 hour data as a forcast

### Import packages and load the data

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import datetime
import plotly.graph_objects as go

In [0]:
data_input_path = '/dbfs/mnt/thesis/output_data/'
prediction_output_path = '/dbfs/mnt/thesis/predictions/benchmark/'

In [0]:
df = pd.read_csv(data_input_path + 'processed_data.csv')
df['DATETIME'] = pd.to_datetime(df['DATETIME'])
df = df.fillna(0)
df.head(3)

,DATETIME,LOCATION,VALUE
0,2009-07-02 00:00:00,0,-79.50
1,2009-07-02 00:05:00,0,-22.81
2,2009-07-02 00:10:00,0,23.02


### Benchmark model

In [0]:
df['DATETIME'] = pd.to_datetime(df['DATETIME'])
df = df.sort_values(['LOCATION', 'DATETIME'])

df['bm_1d_prediction'] = df.groupby('LOCATION')['VALUE'].shift(288)

#forecast range
first_day = pd.Timestamp('2010-08-28')
last_day = pd.Timestamp('2010-08-29')

mask = (df['DATETIME'] >= first_day) & (df['DATETIME'] < last_day)
all_1d_pred = df.loc[mask, ['DATETIME', 'LOCATION', 'VALUE', 'bm_1d_prediction']]
print("EVAL PERIOD", all_1d_pred['DATETIME'].min(), all_1d_pred['DATETIME'].max())
all_1d_pred.to_csv(prediction_output_path + f"predictions_{first_day.date()}.csv", index=False)
all_1d_pred.head(3)

EVAL PERIOD 2010-08-28 00:00:00 2010-08-28 23:55:00


,DATETIME,LOCATION,VALUE,bm_1d_prediction
121536,2010-08-28 00:00:00,0,283.77,151.18
121537,2010-08-28 00:05:00,0,310.48,180.07
121538,2010-08-28 00:10:00,0,317.08,180.13


In [0]:
def plot_predictions(location):
    filtered_df = all_1d_pred[all_1d_pred['LOCATION'] == location]
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(x=filtered_df['DATETIME'], y=filtered_df['VALUE'], mode='lines', name='Actual Value'))
    fig.add_trace(go.Scatter(x=filtered_df['DATETIME'], y=filtered_df['bm_1d_prediction'], mode='lines', name='1 Day Prediction'))
    
    fig.update_layout(title=f'Predictions for Location {location}',
                      xaxis_title='Datetime',
                      yaxis_title='Values')
    
    fig.show()

plot_predictions(3)